In [20]:
import numpy as np
import pandas as pd
import re
import nltk
import pickle
from nltk.corpus import stopwords 
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score









In [21]:
# WNLOAD DEPENDENCIES & INITIALIZE STEMMER


nltk.download('stopwords')
stremmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\VENGEANCE\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
# LOAD DATASET AND EXTRACT 


dataset_path = r"G:\Jupyter-NoteBook\sentimental-analysis\tweeter-dataset.zip"

print(" Loading dataset...")
raw_data = pd.read_csv(dataset_path, compression='zip', encoding='latin-1', header=None)

# Format columns matching your project setup
col_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
raw_data.columns = col_names

# Map positive target class 4 to 1
raw_data['target'] = raw_data['target'].map({4: 1, 0: 0})

# Separate the dataset by classes to select sequentially from the top of each
negative_tweets = raw_data[raw_data['target'] == 0]
positive_tweets = raw_data[raw_data['target'] == 1]

# Take the first 2,500 negative rows and first 2,500 positive rows sequentially
data_negative_segment = negative_tweets.iloc[:2500]
data_positive_segment = positive_tweets.iloc[:2500]

# Combine them into a single dataset of exactly 5,000 rows
data = pd.concat([data_negative_segment, data_positive_segment]).copy()
print(f" Dataset prepared sequentially. Rows: {data.shape[0]} | Target Classes Present: {set(data['target'])}")




 Loading dataset...
 Dataset prepared sequentially. Rows: 5000 | Target Classes Present: {0, 1}


In [23]:
# TEXT PREPROCESSING FUNCTION (STEMMING)


def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content) 
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [stremmer.stem(word) for word in stemmed_content if word not in stop_words]
    return ' '.join(stemmed_content)

print(" Stemming text column (this may take a moment)...")
data['text'] = data['text'].apply(stemming)


 Stemming text column (this may take a moment)...


In [24]:

# SPLIT DATA (WITH SHUFFLE & STRATIFICATION)


x = data['text']
y = data['target']

In [25]:
# Split the dataset safely so training blocks are evenly distributed
x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, 
    shuffle=True, 
    stratify=y, 
    random_state=42
)


In [26]:
# CONVERT TEXT DATA TO NUMBERS (VECTORIZATION)


print(" Vectorizing text data to numbers...")
vectorizer = TfidfVectorizer()

# Compute vocabulary using only train set raw text, transform both sets to arrays
x_train = vectorizer.fit_transform(x_train_raw)
x_test = vectorizer.transform(x_test_raw)


 Vectorizing text data to numbers...


In [27]:

# MODEL TRAINING (LOGISTIC REGRESSION)


print(" Training Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)
print(" Model trained successfully!")


 Training Logistic Regression model...
 Model trained successfully!


In [28]:
# MODEL EVALUATION


y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print(f" Project Model Accuracy: {accuracy:.2%}")


 Project Model Accuracy: 70.40%


In [31]:

# LIVE PREDICTION WRAPPER & TESTS


def predict_sentiment(text):
    text_cleaned = re.sub('[^a-zA-Z]', ' ', text)
    text_cleaned = text_cleaned.lower()
    text_cleaned = text_cleaned.split() 
    text_cleaned = [stremmer.stem(word) for word in text_cleaned if word not in stop_words]
    text_cleaned = ' '.join(text_cleaned)
     # Vectorize input utilizing the fitted TfidfVectorizer instance
    vectorized_input = vectorizer.transform([text_cleaned])
    sentiment = model.predict(vectorized_input)[0]
    
    return "Negative" if sentiment == 0 else "Positive"

print("\n Testing live text assertions:")
print(f"Text: 'I hate you' -> Predicted Sentiment: {predict_sentiment('I hate you')}")
print(f"Text: 'I love you' -> Predicted Sentiment: {predict_sentiment('I love you')}")


    


 Testing live text assertions:
Text: 'I hate you' -> Predicted Sentiment: Negative
Text: 'I love you' -> Predicted Sentiment: Positive


In [32]:
# EXPORT ARTIFACTS FOR STREAMLIT APP


with open('model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)

with open('vectorizer.pkl', 'wb') as vec_file:
    pickle.dump(vectorizer, vec_file)

print("\n Saved updated 'model.pkl' and 'vectorizer.pkl' to project environment.")


 Saved updated 'model.pkl' and 'vectorizer.pkl' to project environment.


In [ ]:
data.head(700)

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,nationwideclass behav mad see
...,...,...,...,...,...,...
695,0,1467985259,Mon Apr 06 23:07:25 PDT 2009,NO_QUERY,ikimb0,tweet arent go
696,0,1467985560,Mon Apr 06 23:07:30 PDT 2009,NO_QUERY,arielk,finish delici breakfast last pari miss milk eu...
697,0,1467986058,Mon Apr 06 23:07:39 PDT 2009,NO_QUERY,robkellas,ilearn great consid final week
698,0,1467986061,Mon Apr 06 23:07:39 PDT 2009,NO_QUERY,harmonicait,carmonium stress outttt
